# Find data worth downloading

Science data products stay on the science server; this database tells you which ones are worth pulling. Once quality reports are ingested, every successful observation carries its executed window and the server-side file names in its `executed` block.

The repo database has no verdicts yet (no real quality reports have flowed), so the first queries return empty frames. The last section fakes one report in a throwaway directory to show the full loop.

In [1]:
import pandas as pd

from pandoraobservations.cache import load_observations

observations = load_observations()
good = observations[observations["verdict"].isin(["success", "partial"])]
good[["target", "executed_start_utc", "executed_stop_utc", "verdict", "overall_score"]]

,target,executed_start_utc,executed_stop_utc,verdict,overall_score


File names live in the record layer (the cache keeps one flat row per observation and does not carry lists). Collecting them:

In [2]:
from pandoraobservations.database import ObservationDatabase


def data_products(db):
    rows = []
    for _, record in db.iter_records("calendars"):
        for obs in record["observations"]:
            quality = obs.get("quality")
            if obs.get("executed") and quality and quality["verdict"] in ("success", "partial"):
                for product in obs["executed"]["data_products"]:
                    rows.append(
                        {
                            "target": obs["target"],
                            "verdict": quality["verdict"],
                            "start_utc": obs["executed"]["start_utc"],
                            "file": product,
                        }
                    )
    return pd.DataFrame(rows)


data_products(ObservationDatabase())

""


## Demonstration with a synthetic report

Everything below happens in a temporary directory; the real database is untouched.

In [3]:
import json
import tempfile
from pathlib import Path

from pandoraobservations.calendars import ingest_calendar
from pandoraobservations.database import init_data_dir
from pandoraobservations.reports import ingest_report

repo = Path.cwd().parent
demo_dir = init_data_dir(Path(tempfile.mkdtemp()) / "data")
ingest_calendar(repo / "examples" / "PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.xml", data_dir=demo_dir)

report = {
    "schema_version": 1,
    "report_id": "quality-demo",
    "generated_utc": "2026-09-01T12:00:00Z",
    "producer": {"name": "demo", "version": "0"},
    "coverage": {"start_utc": "2026-08-24T00:00:00Z", "stop_utc": "2026-08-31T00:00:00Z"},
    "complete": False,
    "observations": [
        {
            "target": "G4476152832143994112",
            "start_utc": "2026-08-24T00:14:00Z",
            "stop_utc": "2026-08-24T00:30:00Z",
            "data_products": ["pan_vda_20260824T001400.fits", "pan_nirda_20260824T001400.fits"],
            "metrics": {
                "pointing_rms_arcsec": {"value": 0.4},
                "data_completeness_frac": {"value": 0.999},
            },
        }
    ],
}
report_path = Path(tempfile.mkdtemp()) / "quality-demo.json"
report_path.write_text(json.dumps(report))
ingest_report(report_path, data_dir=demo_dir)

data_products(ObservationDatabase(demo_dir))

2026-08-21 16:22:52 WARNING: PAN-SCICAL-SCI-20260819-VF-20260824-EX-20260831-R002.xml claims 26 visits / 227       
sequences but contains 25 / 226; recording both.

,target,verdict,start_utc,file
0,G4476152832143994112,success,2026-08-24T00:14:00Z,pan_vda_20260824T001400.fits
1,G4476152832143994112,success,2026-08-24T00:14:00Z,pan_nirda_20260824T001400.fits
